In [ ]:
import zipfile
import os
import warnings
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
warnings.filterwarnings("ignore")
import pandas as pd
from huggingface_hub import hf_hub_download
from datasets import Dataset, DatasetDict, Features, Value, ClassLabel

# 1. Download the raw zip from the repo (this file is on main)
zip_path = hf_hub_download(
    repo_id="takala/financial_phrasebank",
    repo_type="dataset",
    filename="data/FinancialPhraseBank-v1.0.zip",
)

# 2. Find and read the "all agree" file inside the zip
with zipfile.ZipFile(zip_path) as z:
    name = next(n for n in z.namelist() if n.lower().endswith("sentences_50agree.txt"))
    raw = z.read(name).decode("iso-8859-1")

# 3. Parse "sentence@label" lines (split on the LAST "@", since sentences can contain "@")
label_names = ["negative", "neutral", "positive"]
label2id = {n: i for i, n in enumerate(label_names)}

rows = []
for line in raw.splitlines():
    line = line.strip()
    if not line:
        continue
    sentence, label = line.rsplit("@", 1)
    rows.append({"sentence": sentence, "label": label2id[label.strip()]})

# 4. Build a Hugging Face Dataset with the same schema as the original
features = Features({
    "sentence": Value("string"),
    "label": ClassLabel(names=label_names),
})
ds = DatasetDict({"train": Dataset.from_pandas(pd.DataFrame(rows), features=features)})

print(ds)
print(ds["train"][0])
print(ds["train"][1])
print(len(ds["train"]))  

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label'],
        num_rows: 4846
    })
})
{'sentence': 'According to Gran , the company has no plans to move all production to Russia , although that is where the company is growing .', 'label': 1}
{'sentence': 'Technopolis plans to develop in stages an area of no less than 100,000 square meters in order to host companies working in computer technologies and telecommunications , the statement said .', 'label': 1}
4846


In [13]:
#Clean up the DATA set using bag of words and remove stop words and punctuation
#imports for cleaning data
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from tqdm import tqdm
import numpy as np
import re
from matplotlib import pyplot as plt
from nltk.corpus import treebank
from gensim.models import KeyedVectors
from keras_preprocessing.text import Tokenizer
from keras.utils import pad_sequences
from keras.utils import to_categorical
from keras.models import Sequential
from keras.layers import Embedding
from keras.layers import Dense, Input
from keras.layers import TimeDistributed
from keras.layers import LSTM, GRU, SimpleRNN
from keras.models import Model
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
import pandas as pd
import nltk
import tensorflow as tf